In [1]:
import os
import json
import warnings
import pandas as pd
from monty.serialization import loadfn
from pymatgen.core import Structure
from pymatgen.analysis.elasticity import (
    ElasticTensor,
    ElasticTensorExpansion,
)
warnings.filterwarnings("ignore")

In [2]:
files_to_read = [name for name in os.listdir() if ".json" in name]

In [3]:
df = pd.DataFrame(index=["Ag8SiS6", "Ag8GeS6", "Ag8SnS6_RT", "Ag8SnS6_LT"],
                 columns=["v_l", "v_t", "v_m", "kappa_cahill", "kappa_agne"])
df.index.name = 'Composition'

In [4]:
for file in files_to_read:
    composition = file.split(".")[0]
    
    elasticity_data = loadfn(file) # load atomate2 output document from elasticity workflow
    
    etr = ElasticTensorExpansion.from_diff_fit(strains=elasticity_data.fitting_data.strains,
                                     stresses=elasticity_data.fitting_data.pk_stresses,
                                     eq_stress=elasticity_data.eq_stress,
                                     order=elasticity_data.order)
    structure = elasticity_data.structure
    ieee = etr.convert_to_ieee(structure)
    
    property_tensor = ElasticTensor(ieee[0])
    
    v_l = property_tensor.long_v(structure)
    v_t = property_tensor.trans_v(structure)
    df.loc[composition, "v_l"] = v_l
    df.loc[composition, "v_t"] = v_t
    df.loc[composition, "v_m"] = ((1.0/3.0)*(v_l**(-3.0)+2.0*v_t**(-3.0)))**(-1.0/3.0) # compute mean velocity using formula S2 of SI
    df.loc[composition, "kappa_cahill"] = property_tensor.cahill_thermalcond(structure)
    df.loc[composition, "kappa_agne"] = property_tensor.agne_diffusive_thermalcond(structure)

In [5]:
df

,v_l,v_t,v_m,kappa_cahill,kappa_agne
Composition,,,,,
Ag8SiS6,2982.681712,1389.147706,1564.270446,0.428487,0.269204
Ag8GeS6,2960.669508,1368.950475,1542.058176,0.419574,0.263604
Ag8SnS6_RT,2923.27765,1332.495544,1501.982987,0.402087,0.252618
Ag8SnS6_LT,3042.834191,1417.595755,1596.281349,0.421319,0.264701


In [6]:
df.to_csv("sound_velocity_kappa.csv") # dump the velocity (longitudinal, transverse, mean) and kappa(cahill, agne)